# INF6422E – Advanced Concepts in Computer Security  
## Practical Work 2 – Winter 2026  

### Adversarial Machine Learning and Robustness Evaluation

--- 
  
## Students  
- Antoine Khoueiry – *Matricule:* 2487137  
- Louis Junior Mpandzo-Otiankouya – *Matricule:* 2421549  

---

## Objective

This lab explores adversarial machine learning, where attackers manipulate input data in order to fool machine learning–based security systems. We implement adversarial attacks and defenses, and quantitatively evaluate the robustness of deep learning models trained on the CIFAR-10 dataset.  
Adversarial threats are highly relevant in cybersecurity, as modern intrusion detection systems, malware classifiers, and biometric authentication models increasingly rely on machine learning. Understanding robustness is therefore essential for deploying trustworthy AI-powered security solutions.

---

## Dataset

**CIFAR-10**: 60,000 colour images (32×32) across 10 classes, with 6,000 images per class.  
Source: https://www.cs.toronto.edu/~kriz/cifar.html

---

## Notebook Structure

1. Baseline Machine Learning Classification Model  
2. Adversarial Evasion Attacks (Test-Time Threats)  
3. Data Poisoning Attacks (Training-Time Threats)  
4. Defenses and Robustness Trade-Offs


# 1. Baseline Machine Learning Classification Model

## 1.1 Dataset Preparation and Training Pipeline 


In [31]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import defaultdict
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, ConcatDataset, random_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import label_binarize

# ── Device ───────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define data directories
data_dir = r'Datasets'

# Data transformation
data_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),  # Adding vertical flip
    transforms.RandomRotation(20),    # Adding random rotation
    transforms.RandomAffine(15),      # Adding random affine transformation
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.5),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

Using device: cpu


In [32]:
# Load datasets
full_train = datasets.CIFAR10(root=data_dir, train=True, download=True, transform=data_transform)
test_dataset  = datasets.CIFAR10(root=data_dir, train=False, download=True, transform=data_transform)
combined_dataset = ConcatDataset([full_train, test_dataset])

# Split the dataset into train, validation, and test sets
train_size = int(0.7 * len(combined_dataset))
val_size = int(0.15 * len(combined_dataset))
test_size = len(combined_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(combined_dataset, [train_size, val_size, test_size])

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Train : {len(train_dataset):>6} images")
print(f"Val   : {len(val_dataset):>6} images")
print(f"Test  : {len(test_dataset):>6} images")

Files already downloaded and verified
Files already downloaded and verified
Train :  42000 images
Val   :   9000 images
Test  :   9000 images


## 1.2 CNN Model Training

In [33]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3,   out_channels=32,  kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32,  out_channels=64,  kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(in_channels=64,  out_channels=128, kernel_size=3, stride=1, padding=1)
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1)
        self.pool    = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1     = nn.Linear(256 * 2 * 2, 512)
        self.fc2     = nn.Linear(512, num_classes)
        self.dropout = nn.Dropout(0.5)
        self.relu    = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))   # 32×32 → 16×16
        x = self.pool(self.relu(self.conv2(x)))   # 16×16 →  8×8
        x = self.pool(self.relu(self.conv3(x)))   #  8×8  →  4×4
        x = self.pool(self.relu(self.conv4(x)))   #  4×4  →  2×2
        x = x.view(x.size(0), -1)                 # flatten → 256*2*2 = 1024
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

model = SimpleCNN(num_classes=10).to(device)
print(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

SimpleCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=1024, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=10, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (relu): ReLU()
)

Trainable parameters: 918,346


In [34]:
model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 10
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

train_losses = []
val_losses = []
val_accuracies = []

best_val_acc  = 0.0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)  # Move data to GPU
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    class_correct = defaultdict(int)
    class_total = defaultdict(int)

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)  # Move data to GPU
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            for i in range(len(labels)):
                label = labels[i].item()
                pred = predicted[i].item()
                if label == pred:
                    class_correct[label] += 1
                class_total[label] += 1

    val_loss = val_loss / len(val_loader)
    val_losses.append(val_loss)

    val_accuracy = 100 * correct / total
    val_accuracies.append(val_accuracy)

    # Save best model
    if val_accuracy > best_val_acc:
        best_val_acc = val_accuracy
        torch.save(model.state_dict(), 'best_model.pth')

    scheduler.step()

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%")

    for class_idx in range(len(class_total)):
        accuracy = 100 * class_correct[class_idx] / class_total[class_idx]
        print(f"Validation Accuracy for class {class_idx}: {accuracy:.2f}%")

print(f"\nBest validation accuracy: {best_val_acc:.2f}%")


Epoch 1/10, Train Loss: 1.9951, Val Loss: 1.7927, Val Accuracy: 34.38%
Validation Accuracy for class 0: 34.53%
Validation Accuracy for class 1: 30.64%
Validation Accuracy for class 2: 14.32%
Validation Accuracy for class 3: 27.49%
Validation Accuracy for class 4: 30.33%
Validation Accuracy for class 5: 19.89%
Validation Accuracy for class 6: 60.34%
Validation Accuracy for class 7: 45.91%
Validation Accuracy for class 8: 31.26%
Validation Accuracy for class 9: 50.22%
Epoch 2/10, Train Loss: 1.7097, Val Loss: 1.6198, Val Accuracy: 42.01%
Validation Accuracy for class 0: 42.29%
Validation Accuracy for class 1: 47.02%
Validation Accuracy for class 2: 38.53%
Validation Accuracy for class 3: 21.69%
Validation Accuracy for class 4: 28.41%
Validation Accuracy for class 5: 53.95%
Validation Accuracy for class 6: 43.54%
Validation Accuracy for class 7: 44.30%
Validation Accuracy for class 8: 52.77%
Validation Accuracy for class 9: 46.86%
Epoch 3/10, Train Loss: 1.5963, Val Loss: 1.5353, Val Accu

In [36]:
# Load the trained model
model = SimpleCNN(num_classes=10)
model.load_state_dict(torch.load('best_model.pth'))

# Check if GPU is available and move the model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

model.eval()

# Evaluate on the test set
correct = 0
total = 0
class_correct = defaultdict(int)
class_total = defaultdict(int)

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)  # Move data to GPU
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        for i in range(len(labels)):
            label = labels[i].item()
            pred = predicted[i].item()
            if label == pred:
                class_correct[label] += 1
            class_total[label] += 1

test_accuracy = 100 * correct / total
print(f"Test Accuracy: {test_accuracy:.2f}%")

for class_idx in range(len(class_total)):
    accuracy = 100 * class_correct[class_idx] / class_total[class_idx]
    print(f"Test Accuracy for class {class_idx}: {accuracy:.2f}%")

C:\Users\louis\AppData\Local\Temp\ipykernel_53784\2545294249.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model.pth'))


Test Accuracy: 57.22%
Test Accuracy for class 0: 62.54%
Test Accuracy for class 1: 74.40%
Test Accuracy for class 2: 39.43%
Test Accuracy for class 3: 26.54%
Test Accuracy for class 4: 54.10%
Test Accuracy for class 5: 49.03%
Test Accuracy for class 6: 69.18%
Test Accuracy for class 7: 58.05%
Test Accuracy for class 8: 72.55%
Test Accuracy for class 9: 65.08%


# 2. Adversarial Evasion Attacks (Test-Time Threats)

## 2.1 FGSM Attack Implementation


## 2.2 PGD Attack Implementation

# 3. Data Poisoning Attacks (Training-Time Threats)

## 3.1 Label-Flipping Poisoning Experiment


## 3.2 Quantitative Poisoning Impact

# 4. Baseline Machine Learning Classification Model

## 4.1 Dataset Preparation and Training Pipeline 
